# LangChain Resume Screening & Ranking

Screens resumes against a job description with an LLM (OpenAI, Gemini, or Cohere) and ranks candidates.

**Pipeline**
1. Verify API connectivity (LLM, optionally embeddings)
2. Load JD + resumes (`PyPDFLoader` / text), redact contact info
3. *Optional* embedding pre-filter for large pools (chunk with `RecursiveCharacterTextSplitter`, keep top N)
4. LangChain chain: `ChatPromptTemplate | llm.with_structured_output(Screening) | retry`, run with `.batch()` and bounded concurrency
5. Weighted score computed in code, missing must-haves penalized
6. Disk cache so reruns skip already-screened resumes
7. Rank and export

In [ ]:
# Already installed per your environment. Optional: DOCX support
# %pip install -q python-docx

In [1]:
# Check package version

import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community",  "langchain-text-splitters",
            "langchain-openai", "langchain-google-genai", "langchain-cohere",
            "pypdf"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-text-splitters version: 1.1.2
langchain-openai version: 1.2.2
langchain-google-genai version: 4.4.0
langchain-cohere version: 0.6.0
pypdf version: 6.10.2


## 1. Configuration

In [2]:
import os, re, json, hashlib, getpass
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, field_validator
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Ash\AppData\Local\Temp\ipykernel_2716\3960406339.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
PROVIDER = "gemini"   # Here options are =  "openai" | "gemini" | "cohere"

LLM_MODELS = {
    "openai": "gpt-4o-mini",
    "gemini": "gemini-2.5-flash",
    "cohere": "command-a-03-2025",
}
EMBED_MODELS = {
    "openai": "text-embedding-3-small",
    "gemini": "gemini-embedding-001",
    "cohere": "embed-english-v3.0",
}
KEY_ENV = {"openai": "OPENAI_API_KEY", "gemini": "GOOGLE_API_KEY", "cohere": "COHERE_API_KEY"}


In [8]:

# Scoring (weights must sum to 1.0)
WEIGHTS = {"skills_match": 0.40, "experience_relevance": 0.30, "achievements": 0.15, "education": 0.15}
MISSING_MUST_HAVE_PENALTY = 10     # points per missing must-have

# Scale controls
MAX_RESUME_CHARS = 15_000
MAX_CONCURRENCY = 5                # parallel LLM calls
BATCH_SIZE = 50                    # resumes per .batch() call (progress + failure isolation)
MAX_RETRIES = 3
USE_CACHE = True
CACHE_DIR = Path(".screening_cache")

# Embedding pre-filter: applies only when pool > PREFILTER_MIN_RESUMES
PREFILTER_ENABLED = True
PREFILTER_MIN_RESUMES = 50
PREFILTER_TOP_N = 25

RESUME_DIR = Path("resumes")           # .pdf / .txt / .md / .docx
JD_PATH = Path("job_description.txt")
OUTPUT_CSV = "ranked_candidates.csv"

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9
CACHE_DIR.mkdir(exist_ok=True)

## Model factories

In [9]:
def get_llm(provider: str = PROVIDER):
    model = LLM_MODELS[provider]
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model, temperature=0)
    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model, temperature=0)
    if provider == "cohere":
        from langchain_cohere import ChatCohere
        return ChatCohere(model=model, temperature=0)
    raise ValueError(f"Unknown provider: {provider}")

def get_embeddings(provider: str = PROVIDER):
    model = EMBED_MODELS[provider]
    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=model)
    if provider == "gemini":
        from langchain_google_genai import GoogleGenerativeAIEmbeddings
        return GoogleGenerativeAIEmbeddings(model=model)
    if provider == "cohere":
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(model=model)
    raise ValueError(f"Unknown provider: {provider}")

##  API connection test
Sends a minimal request to the LLM (and embeddings, if the pre-filter is enabled) and reports a clear pass/fail per provider.

In [10]:
def test_connection(provider: str, check_embeddings: bool = False) -> tuple[bool, str]:
    if not os.environ.get(KEY_ENV[provider]):
        return False, f"{KEY_ENV[provider]} not set"
    try:
        reply = get_llm(provider).invoke("Reply with the single word: OK")
        detail = f"LLM {LLM_MODELS[provider]} -> {str(reply.text)[:30]!r}"
        if check_embeddings:
            dim = len(get_embeddings(provider).embed_query("ping"))
            detail += f"; embeddings {EMBED_MODELS[provider]} dim={dim}"
        return True, detail
    except Exception as e:
        msg = str(e).replace("\n", " ")[:250]
        hint = ""
        low = msg.lower()
        if any(s in low for s in ("401", "403", "api key", "api_key", "unauthorized", "permission", "invalid")):
            hint = " [likely invalid/unauthorized key]"
        elif any(s in low for s in ("404", "not found", "model")):
            hint = " [check model name/access]"
        elif any(s in low for s in ("429", "quota", "rate")):
            hint = " [rate limit/quota]"
        return False, f"{type(e).__name__}: {msg}{hint}"

def ensure_key(provider: str):
    if not os.environ.get(KEY_ENV[provider]):
        os.environ[KEY_ENV[provider]] = getpass.getpass(f"{KEY_ENV[provider]}: ").strip()

# Optional: check every provider whose key is already in the environment
for p in KEY_ENV:
    if os.environ.get(KEY_ENV[p]):
        ok, msg = test_connection(p)
        print(f"{'OK  ' if ok else 'FAIL'} {p:7s} {msg}")

In [11]:
ensure_key(PROVIDER)
ok, msg = test_connection(PROVIDER, check_embeddings=PREFILTER_ENABLED)
print(f"{'OK  ' if ok else 'FAIL'} {PROVIDER}: {msg}")
if not ok:
    raise RuntimeError(f"Connection test failed for {PROVIDER}: {msg}")

GOOGLE_API_KEY:  ········


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


OK   gemini: LLM gemini-2.5-flash -> 'OK'; embeddings gemini-embedding-001 dim=3072


## 2. Output schema

In [12]:
class Screening(BaseModel):
    candidate_name: str = Field(description="Candidate's name as written on the resume")
    skills_match: int = Field(description="0-100: coverage of required and preferred skills")
    experience_relevance: int = Field(description="0-100: relevance and depth of experience vs. role")
    achievements: int = Field(description="0-100: measurable impact and accomplishments")
    education: int = Field(description="0-100: fit with education requirements")
    must_have_missing: list[str] = Field(description="Required JD items not demonstrated in the resume")
    strengths: list[str] = Field(description="Max 4 strengths relevant to the role")
    gaps: list[str] = Field(description="Max 4 gaps relevant to the role")
    summary: str = Field(description="Max 2 sentences")
    recommendation: Literal["strong_yes", "yes", "maybe", "no"]

    @field_validator("skills_match", "experience_relevance", "achievements", "education", mode="before")
    @classmethod
    def _clamp(cls, v):
        return max(0, min(100, int(v)))

## 5. Load job description and resumes

In [14]:
def read_resume(path: Path) -> str:
    ext = path.suffix.lower()
    if ext == ".pdf":
        return "\n".join(d.page_content for d in PyPDFLoader(str(path)).load())
    if ext == ".docx":
        from docx import Document   # requires python-docx
        return "\n".join(p.text for p in Document(str(path)).paragraphs)
    return path.read_text(errors="ignore")

def redact(text: str) -> str:
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]", text)
    text = re.sub(r"\+?\d[\d\s().-]{8,}\d", "[PHONE]", text)
    return text

SAMPLE_JD = """Senior Python Engineer - Data Platform
Must have: 5+ years Python, SQL, Airflow or similar orchestration, AWS.
Nice to have: Spark, Kafka, dbt, mentoring experience.
Responsibilities: build and maintain batch and streaming data pipelines, improve reliability, collaborate with analysts.
Education: Bachelor's in CS or related field."""

SAMPLE_RESUMES = {
    "alice.txt": "Alice Rao. 7 years Python, SQL, Airflow, AWS (S3, Glue, EMR). \
                  Built Spark pipelines processing 2TB/day, cut runtime 40%. Led 3 juniors. B.Tech CS.",
    
    "bob.txt": "Bob Singh. 2 years Java developer. Some Python scripting, MySQL. \
                Built REST APIs with Spring. B.Sc Physics.",
    
    "carol.txt": "Carol Mehta. 5 years Python and SQL, Airflow DAGs, Kafka streaming, \
                  dbt models on Snowflake. Deployed on GCP, limited AWS exposure. M.S. Computer Science.",
}

jd = JD_PATH.read_text() if JD_PATH.exists() else SAMPLE_JD
files = sorted(p for p in RESUME_DIR.glob("*") if p.suffix.lower() in {".pdf", ".docx", ".txt", ".md"}) if RESUME_DIR.exists() else []

resumes, load_errors = {}, {}
for p in files:
    try:
        resumes[p.name] = redact(read_resume(p))
    except Exception as e:
        load_errors[p.name] = f"{type(e).__name__}: {e}"
if not files:
    resumes = {k: redact(v) for k, v in SAMPLE_RESUMES.items()}

print(f"{len(resumes)} resumes loaded (sample data: {not files}); {len(load_errors)} failed to load")
for k, v in load_errors.items():
    print("  ", k, v)

3 resumes loaded (sample data: True); 0 failed to load


## 6. Embedding pre-filter (large pools only)
Chunks each resume, embeds chunks, scores a resume by the mean of its top-3 chunk similarities to the JD, and keeps the top N for LLM screening.

In [15]:
def prefilter(resumes: dict, jd: str, top_n: int) -> tuple[dict, dict]:
    emb = get_embeddings(PROVIDER)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

    chunks, owners = [], []
    for name, text in resumes.items():
        for c in splitter.split_text(text[:MAX_RESUME_CHARS]):
            chunks.append(c); owners.append(name)

    q = np.array(emb.embed_query(jd))
    vecs = np.array(emb.embed_documents(chunks))
    sims = vecs @ q / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(q) + 1e-9)

    per_resume = {}
    for name, s in zip(owners, sims):
        per_resume.setdefault(name, []).append(s)
    score = {n: float(np.mean(sorted(v, reverse=True)[:3])) for n, v in per_resume.items()}

    keep = sorted(score, key=score.get, reverse=True)[:top_n]
    print(f"Pre-filter: {len(resumes)} -> {len(keep)} resumes")
    return {n: resumes[n] for n in keep}, score

prefilter_scores = {}
if PREFILTER_ENABLED and len(resumes) > PREFILTER_MIN_RESUMES:
    resumes, prefilter_scores = prefilter(resumes, jd, PREFILTER_TOP_N)
else:
    print("Pre-filter skipped")

Pre-filter skipped


## 7. Screening chain

In [16]:
SYSTEM_PROMPT = """

You are an expert technical recruiter. Evaluate the resume strictly against the job description.

Rules:
- Base every judgment only on evidence in the resume. Do not infer skills that are not stated.
- Ignore name, gender, age, nationality, and other protected attributes when scoring.
- Score each criterion 0-100 (0 = no evidence, 50 = partial, 100 = exceeds requirement).
- must_have_missing: only requirements the JD marks as required/must-have that the resume does not demonstrate.
- Be concise."""


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "JOB DESCRIPTION:\n{jd}\n\nRESUME:\n{resume}"),
])

def build_chain():
    structured = get_llm(PROVIDER).with_structured_output(Screening)
    return (prompt | structured).with_retry(stop_after_attempt=MAX_RETRIES, wait_exponential_jitter=True)

def weighted_score(s: Screening) -> float:
    base = sum(getattr(s, k) * w for k, w in WEIGHTS.items())
    return max(0.0, round(base - MISSING_MUST_HAVE_PENALTY * len(s.must_have_missing), 1))

def cache_path(text: str) -> Path:
    key = hashlib.sha256(f"{PROVIDER}|{LLM_MODELS[PROVIDER]}|{jd}|{text}".encode()).hexdigest()
    return CACHE_DIR / f"{key}.json"

def screen_all(resumes: dict):
    chain = build_chain()
    results, errors, todo = {}, {}, []

    for name, text in resumes.items():
        p = cache_path(text)
        if USE_CACHE and p.exists():
            results[name] = Screening.model_validate_json(p.read_text())
        else:
            todo.append((name, text[:MAX_RESUME_CHARS], p))
    print(f"{len(results)} from cache, {len(todo)} to screen")

    for i in range(0, len(todo), BATCH_SIZE):
        chunk = todo[i:i + BATCH_SIZE]
        outs = chain.batch(
            [{"jd": jd, "resume": t} for _, t, _ in chunk],
            config={"max_concurrency": MAX_CONCURRENCY},
            return_exceptions=True,
        )
        for (name, _, p), o in zip(chunk, outs):
            if isinstance(o, Screening):
                results[name] = o
                p.write_text(o.model_dump_json())
            else:
                errors[name] = f"{type(o).__name__}: {o}" if o is not None else "Empty/unparseable model output"
        print(f"  processed {min(i + BATCH_SIZE, len(todo))}/{len(todo)}")
    return results, errors

## 8. Run

In [17]:
results, errors = screen_all(resumes)

rows = [{"file": n, **s.model_dump(), "overall": weighted_score(s)} for n, s in results.items()]
if prefilter_scores:
    for r in rows:
        r["prefilter_sim"] = round(prefilter_scores.get(r["file"], float("nan")), 4)
df = pd.DataFrame(rows)

if errors:
    print("Failed:")
    for k, v in errors.items():
        print(f"  {k}: {v[:200]}")

0 from cache, 3 to screen
  processed 3/3


## 9. Ranking

In [18]:
ranked = df.sort_values(["overall", "skills_match"], ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

pd.set_option("display.max_colwidth", 80)
ranked[["rank", "candidate_name", "overall", "skills_match", "experience_relevance",
        "achievements", "education", "recommendation", "must_have_missing"]]

,rank,candidate_name,overall,skills_match,experience_relevance,achievements,education,recommendation,must_have_missing
0,1,Alice Rao,94.5,90,95,100,100,strong_yes,[]
1,2,Carol Mehta,56.0,60,90,0,100,maybe,[AWS (limited exposure for a must-have)]
2,3,Bob Singh,0.0,10,5,0,70,no,"[5+ years Python experience, Airflow or similar orchestration, AWS, Experien..."


In [19]:
for _, r in ranked.iterrows():
    print(f"#{r['rank']} {r['candidate_name']} ({r['overall']}) -> {r['recommendation']}")
    print(f"   {r['summary']}")
    print(f"   Strengths: {'; '.join(r['strengths'])}")
    print(f"   Gaps: {'; '.join(r['gaps'])}")
    if r["must_have_missing"]:
        print(f"   Missing must-haves: {', '.join(r['must_have_missing'])}")
    print()

#1 Alice Rao (94.5) -> strong_yes
   Alice Rao is a highly experienced Senior Python Engineer with 7 years in data platform development, demonstrating strong skills in Python, SQL, Airflow, and AWS. She has a proven track record of building and optimizing large-scale data pipelines and leading junior team members.
   Strengths: Extensive Python, SQL, Airflow, and AWS experience; Proven ability to build high-volume data pipelines (2TB/day); Demonstrated impact through performance optimization (40% runtime reduction); Experience leading and mentoring junior engineers
   Gaps: No explicit mention of Kafka; No explicit mention of dbt; No explicit mention of collaborating with analysts

#2 Carol Mehta (56.0) -> maybe
   Carol demonstrates strong technical skills in Python, SQL, Airflow, Kafka, and dbt, supported by an M.S. in Computer Science. However, her limited AWS experience is a significant gap for this role's must-have requirement.
   Strengths: Strong Python and SQL experience (5 yea

## 10. Export

In [20]:
out = ranked.copy()
for c in ["must_have_missing", "strengths", "gaps"]:
    out[c] = out[c].apply("; ".join)
out.to_csv(OUTPUT_CSV, index=False)

if errors or load_errors:
    pd.DataFrame(
        [{"file": k, "stage": "load", "error": v} for k, v in load_errors.items()] +
        [{"file": k, "stage": "screen", "error": v} for k, v in errors.items()]
    ).to_csv("screening_errors.csv", index=False)
print(f"Saved {OUTPUT_CSV}")

Saved ranked_candidates.csv


## Notes
- **Switch provider:** change `PROVIDER` and rerun from the config cell. Keys are read from `OPENAI_API_KEY`, `GOOGLE_API_KEY`, `COHERE_API_KEY`.
- **Scaling:** raise `MAX_CONCURRENCY` until you hit provider rate limits; `with_retry` backs off on transient errors. The cache key includes provider, model, JD, and resume text, so changing any of them triggers a fresh screening.
- **Pre-filter trade-off:** embedding similarity can drop strong candidates with unusual phrasing. Set `PREFILTER_TOP_N` generously.
- **Score stability:** LLM scores vary slightly between runs. For borderline candidates, rescreen the top group or compare pairwise.
- **Compliance:** only email/phone are redacted; names are still sent to the LLM. Use rankings as decision support with human review.

# Student Exercises: Extending the Resume Screening Notebook

**Starting point:** `resume_screening_langchain.ipynb`
**Setup for all tasks:** use 10–20 resumes, keep `USE_CACHE = True`, and use a free-tier key (Gemini or Cohere trial) to stay within quota.

## Level 1: Config and prompt changes (30–60 min each)

| # | Task | What to change | Deliverable |
|---|------|----------------|-------------|
| 1 | **Tune weights** | Edit `WEIGHTS` (e.g., skills 0.6, education 0.05). Rerun. | Table of rank before vs. after, plus 3 sentences on what moved and why |
| 2 | **Compare providers** | Run the same resumes with two `PROVIDER` values. | Side-by-side ranking, and a list of candidates the models disagree on |
| 3 | **Anchor the rubric** | Edit `SYSTEM_PROMPT` to define what 25, 50, 75 and 100 mean for each criterion. | Score distribution (min/max/std) before vs. after |
| 4 | **Extend redaction** | Update `redact()` to also remove the candidate's name and graduation years. | 3 before/after redaction examples |
| 5 | **Build a test set** | Write a JD and 10 resumes: 3 strong, 4 average, 3 irrelevant. | Your `job_description.txt` and `resumes/` folder |

## Level 2: Add code (1–3 hrs each)

| # | Task | Approach | Deliverable |
|---|------|----------|-------------|
| 6 | **JD checklist** | Add a second chain that turns the JD into `must_have` and `nice_to_have` lists (pydantic model). Pass the lists into the scoring prompt. | Checklist output for your JD, and whether `must_have_missing` is now more consistent |
| 7 | **Evidence check** | Add `evidence: list[str]` to `Screening` (a quote per strength). Write a function that checks each quote appears in the resume text (lowercase, normalize whitespace). | Count of quotes that fail the check, with examples |
| 8 | **Accuracy evaluation** | Rank your 10 resumes by hand. Compute Spearman correlation with `scipy.stats.spearmanr` against the notebook's ranking. | Correlation value, and the 2 biggest disagreements explained |
| 9 | **Consistency test** | Set `USE_CACHE = False`. Screen one resume 5 times. | Mean and standard deviation of `overall` |
| 10 | **Bias test** | Take one resume and change only the name (try 4 different names). Screen each. | Score per name, and a conclusion on whether scores changed |
| 11 | **Cost tracking** | Read `usage_metadata` from the model responses (or use `get_usage_metadata_callback`). Sum input and output tokens. | Total tokens and estimated cost per resume |

## Level 3: Challenge (optional)

12. **Hybrid pre-filter:** combine BM25 (`rank_bm25`) with the embedding score, or add `CohereRerank`. Compare which resumes survive vs. the embedding-only filter.
13. **Two-tier screening:** screen everyone with a cheap model, then rescreen the top 5 with a stronger one.
14. **Interview questions:** add a chain that generates 3 questions per candidate based on their `gaps`.
15. **Streamlit app:** upload a JD and resumes, and display the ranked table.
16. **Persistent vector store:** save embeddings in FAISS or Chroma so changing the JD doesn't re-embed resumes.

